# Demo 3: Build past-only features and audit availability

**Learning question:** How can we compare each observation with its own entity's past while rejecting values that were unavailable at a supplied decision time?

The temporal-feature and chronological-block tables preserve the input grain of **one recorded station observation per row**. The availability table has grain one candidate per row. This required demo is Colab-first and runs equivalently in local Jupyter or VS Code. Colab storage is ephemeral, and changes opened from GitHub are not automatically saved back to the repository.

Use only the supplied synthetic, non-identifying fixture. Do not add credentials, private records, manual uploads, or Drive mounts. Restart the kernel and run every cell in order; stored output is not execution evidence. Assignment Colab support remains conditional on the repository-save and Classroom50 pilot.


In [ ]:
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

PYTHON_CANDIDATE = "3.12.13"
NUMPY_CANDIDATE = "2.0.2"
PANDAS_CANDIDATE = "3.0.3"
COURSE_PACKAGES = {"numpy": NUMPY_CANDIDATE, "pandas": PANDAS_CANDIDATE}


def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


mismatched = [
    f"{package_name}=={candidate}"
    for package_name, candidate in COURSE_PACKAGES.items()
    if installed_version(package_name) != candidate
]
if mismatched:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *mismatched]
    )

import numpy as np
import pandas as pd

assert platform.python_version() == PYTHON_CANDIDATE
assert np.__version__ == NUMPY_CANDIDATE
assert pd.__version__ == PANDAS_CANDIDATE
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)


## Define entity-scoped, past-only comparisons

This notebook repeats Demo 1's exact parsing, localization, UTC conversion, and station/time sort so it is independently restartable.

A **lag** attaches an earlier observation from the same entity. A **lead** attaches a later observation and is not computed here. A **difference** subtracts the previous same-entity observation from the current value. `shift(1)` means one previous row, not one elapsed hour.

A **trailing window** summarizes values while moving forward through an ordered history. An **observation-count window** selects a fixed number of rows regardless of spacing. An **elapsed-time window** selects observations whose timestamps fall inside a stated clock interval. Both required windows exclude the current row and remain scoped to `station`.


In [ ]:
from hashlib import sha256
from pathlib import Path

EXPECTED_FIXTURE_SHA256 = "57dcdb82372805cf1dda83a7c227b463fe997cf1437275d64d01b9719ff26b54"
FIXTURE_BYTES = (
    b"station,observed_at,temperature_c\n"
    b"south,2026-01-15 13:00,23.0\n"
    b"north,2026-01-15 08:00,10.0\n"
    b"south,2026-01-15 08:00,20.0\n"
    b"north,2026-01-15 14:00,14.0\n"
    b"south,2026-01-15 10:00,21.0\n"
    b"north,2026-01-15 11:00,\n"
    b"south,2026-01-15 14:00,24.0\n"
    b"north,2026-01-15 09:00,11.0\n"
    b"south,2026-01-15 11:00,22.0\n"
    b"north,2026-01-15 12:00,13.0\n"
)


def find_demo_directory(start):
    current = start.resolve()
    while True:
        for candidate in (current, current / "09" / "demo"):
            if (
                (candidate / "DEMO_GUIDE.md").is_file()
                and (candidate / ".python-version").is_file()
            ):
                return candidate
        if current.parent == current:
            return None
        current = current.parent


DEMO_DIRECTORY = find_demo_directory(Path.cwd())
if DEMO_DIRECTORY is None:
    DEMO_DIRECTORY = Path.cwd().resolve()

DATA_DIRECTORY = DEMO_DIRECTORY / "data"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIXTURE_PATH = DATA_DIRECTORY / "station_observations.csv"
if not FIXTURE_PATH.exists():
    FIXTURE_PATH.write_bytes(FIXTURE_BYTES)

actual_fixture_sha256 = sha256(FIXTURE_PATH.read_bytes()).hexdigest()
assert actual_fixture_sha256 == EXPECTED_FIXTURE_SHA256, (
    "station_observations.csv does not match the supplied fixture checksum. "
    "Restore the committed file; corrupt data are never replaced silently."
)

raw = pd.read_csv(
    FIXTURE_PATH,
    dtype={
        "station": "string",
        "observed_at": "string",
        "temperature_c": "float64",
    },
)
raw["source_row"] = np.int64(1)

assert raw.shape == (10, 4)
assert raw["station"].dtype == pd.StringDtype()
assert raw["observed_at"].dtype == pd.StringDtype()
assert raw["temperature_c"].dtype == np.dtype("float64")
assert raw["source_row"].dtype == np.dtype("int64")
assert raw["temperature_c"].isna().sum() == 1

naive_times = pd.to_datetime(
    raw["observed_at"],
    format="%Y-%m-%d %H:%M",
)
assert naive_times.dt.tz is None
aware_times = naive_times.dt.tz_localize("America/Los_Angeles")
raw["observed_at"] = aware_times.dt.tz_convert("UTC")

prepared = raw.sort_values(
    ["station", "observed_at"],
    kind="stable",
).reset_index(drop=True)

assert prepared["station"].dtype == pd.StringDtype()
assert str(prepared["observed_at"].dtype) == "datetime64[us, UTC]"
assert prepared["temperature_c"].dtype == np.dtype("float64")
assert prepared["source_row"].dtype == np.dtype("int64")
assert prepared.shape == (10, 4)
assert not prepared.duplicated(["station", "observed_at"]).any()
assert all(
    group["observed_at"].is_monotonic_increasing
    for _, group in prepared.groupby(
        "station", observed=True, sort=True, dropna=True
    )
)


OUTPUT_DIRECTORY = DEMO_DIRECTORY / "output"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
OWNED_OUTPUT_NAMES = ['temporal_features.csv', 'availability_decisions.csv', 'chronological_blocks.csv']
for output_name in OWNED_OUTPUT_NAMES:
    output_path = OUTPUT_DIRECTORY / output_name
    if output_path.exists():
        output_path.unlink()


def write_verified_csv(frame, path, *, expected_size, expected_sha256):
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        na_rep="",
    )
    first_bytes = path.read_bytes()
    assert len(first_bytes) == expected_size
    assert sha256(first_bytes).hexdigest() == expected_sha256
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        na_rep="",
    )
    assert path.read_bytes() == first_bytes
    return first_bytes


def restore_utc_timestamp(frame, column="observed_at"):
    frame[column] = pd.to_datetime(
        frame[column],
        format="%Y-%m-%d %H:%M:%S%z",
        utc=True,
    )
    return frame


print("Demo directory:", DEMO_DIRECTORY)
print("Fixture SHA-256:", actual_fixture_sha256)


In [ ]:
features = prepared[["station", "observed_at", "temperature_c"]].copy()
by_station = features.groupby(
    "station",
    observed=True,
    sort=True,
    dropna=True,
)["temperature_c"]
features["temperature_lag_1"] = by_station.shift(1)
features["temperature_difference"] = by_station.diff()
features["mean_previous_2_observations"] = (
    features.groupby(
        "station",
        observed=True,
        sort=True,
        dropna=True,
    )["temperature_c"].transform(
        lambda values: values.shift(1)
        .rolling(window=2, min_periods=1)
        .mean()
    )
)

elapsed_summary = (
    features.set_index("observed_at")
    .groupby(
        "station",
        observed=True,
        sort=True,
        dropna=True,
    )["temperature_c"]
    .rolling("2h", closed="left", min_periods=1)
    .mean()
    .rename("mean_previous_2h")
    .reset_index()
)
features = features.merge(
    elapsed_summary,
    on=["station", "observed_at"],
    how="left",
    validate="one_to_one",
    sort=False,
)

assert features.shape == (10, 7)
assert features["station"].dtype == pd.StringDtype()
assert str(features["observed_at"].dtype) == "datetime64[us, UTC]"
for column in features.columns[2:]:
    assert features[column].dtype == np.dtype("float64"), column
pd.testing.assert_frame_equal(
    features[["station", "observed_at", "temperature_c"]],
    prepared[["station", "observed_at", "temperature_c"]],
)

first_rows = features.groupby(
    "station", observed=True, sort=True, dropna=True
).head(1)
assert first_rows["temperature_lag_1"].isna().all()
assert first_rows["temperature_difference"].isna().all()

north_at_17 = features.loc[
    features["station"].eq("north")
    & features["observed_at"].eq(
        pd.Timestamp("2026-01-15 17:00", tz="UTC")
    )
].iloc[0]
south_at_21 = features.loc[
    features["station"].eq("south")
    & features["observed_at"].eq(
        pd.Timestamp("2026-01-15 21:00", tz="UTC")
    )
].iloc[0]
south_at_22 = features.loc[
    features["station"].eq("south")
    & features["observed_at"].eq(
        pd.Timestamp("2026-01-15 22:00", tz="UTC")
    )
].iloc[0]

np.testing.assert_allclose(
    north_at_17[
        [
            "temperature_lag_1",
            "temperature_difference",
            "mean_previous_2_observations",
            "mean_previous_2h",
        ]
    ].to_numpy(dtype="float64"),
    [10.0, 1.0, 10.0, 10.0],
)
np.testing.assert_allclose(
    south_at_21[
        [
            "temperature_lag_1",
            "temperature_difference",
            "mean_previous_2_observations",
            "mean_previous_2h",
        ]
    ].to_numpy(dtype="float64"),
    [22.0, 1.0, 21.5, 22.0],
)
np.testing.assert_allclose(
    south_at_22[
        ["mean_previous_2_observations", "mean_previous_2h"]
    ].to_numpy(dtype="float64"),
    [22.5, 23.0],
)

print(features)


In [ ]:
FEATURES_OUTPUT_PATH = OUTPUT_DIRECTORY / "temporal_features.csv"
features_bytes = write_verified_csv(
    features,
    FEATURES_OUTPUT_PATH,
    expected_size=633,
    expected_sha256="5a6524e8dbb37da3cc056cc648e5ff444c12cb485214bef1a95c3a67a22af3ab",
)
features_readback = pd.read_csv(
    FEATURES_OUTPUT_PATH,
    dtype={
        "station": "string",
        "observed_at": "string",
        "temperature_c": "float64",
        "temperature_lag_1": "float64",
        "temperature_difference": "float64",
        "mean_previous_2_observations": "float64",
        "mean_previous_2h": "float64",
    },
)
restore_utc_timestamp(features_readback)
assert features_readback["station"].dtype == pd.StringDtype()
assert str(features_readback["observed_at"].dtype) == "datetime64[us, UTC]"
for column in features_readback.columns[2:]:
    assert features_readback[column].dtype == np.dtype("float64"), column
pd.testing.assert_frame_equal(features_readback, features)

print("Wrote:", FEATURES_OUTPUT_PATH)
print("Features SHA-256:", sha256(features_bytes).hexdigest())


## Inventory information before accepting a candidate

A **candidate feature** is a value that might later be supplied to a prediction procedure; Lecture 10 formalizes that role. The **prediction timestamp** is the supplied instant when the prediction would be issued. **Information availability** asks whether every required source value was known by that instant.

A **future-derived candidate** requires a value recorded after the prediction timestamp. Treating it as already known creates **future leakage**. A centered window uses observations on both sides of a row. The inventory below names centered and next-observation candidates only to reject them; neither is computed.

A **chronological holdout** sets aside a later time block while work uses an earlier block. This notebook checks only chronological plausibility. Lecture 10 still owns targets, horizons, formal evaluation roles, baselines, metrics, and model selection.


In [ ]:
prediction_timestamp = pd.Timestamp("2026-01-15 21:00", tz="UTC")
availability = pd.DataFrame(
    {
        "candidate": pd.Series(
            [
                "calendar hour",
                "previous observed temperature",
                "centered three-observation mean",
                "next observed temperature",
            ],
            dtype="string",
        ),
        "latest_required_timestamp": pd.to_datetime(
            [
                "2026-01-15 21:00Z",
                "2026-01-15 19:00Z",
                "2026-01-15 22:00Z",
                "2026-01-15 22:00Z",
            ],
            utc=True,
        ),
    }
)
availability["available_by_prediction_time"] = availability[
    "latest_required_timestamp"
].le(prediction_timestamp)
availability["decision"] = pd.Series(
    np.where(
        availability["available_by_prediction_time"],
        "keep",
        "reject",
    ),
    dtype="string",
)

assert availability.shape == (4, 4)
assert availability["candidate"].dtype == pd.StringDtype()
assert str(availability["latest_required_timestamp"].dtype) == "datetime64[us, UTC]"
assert availability["available_by_prediction_time"].dtype == np.dtype("bool")
assert availability["decision"].dtype == pd.StringDtype()
assert availability["available_by_prediction_time"].tolist() == [
    True,
    True,
    False,
    False,
]
assert availability["decision"].tolist() == [
    "keep",
    "keep",
    "reject",
    "reject",
]

AVAILABILITY_OUTPUT_PATH = OUTPUT_DIRECTORY / "availability_decisions.csv"
availability_bytes = write_verified_csv(
    availability,
    AVAILABILITY_OUTPUT_PATH,
    expected_size=326,
    expected_sha256="d4125def8dcf8e23b9f33574f1dd9e14a5ed3f92889f88b455509110ad87e505",
)
availability_readback = pd.read_csv(
    AVAILABILITY_OUTPUT_PATH,
    dtype={
        "candidate": "string",
        "latest_required_timestamp": "string",
        "available_by_prediction_time": "bool",
        "decision": "string",
    },
)
restore_utc_timestamp(availability_readback, "latest_required_timestamp")
assert availability_readback["candidate"].dtype == pd.StringDtype()
assert str(availability_readback["latest_required_timestamp"].dtype) == "datetime64[us, UTC]"
assert availability_readback["available_by_prediction_time"].dtype == np.dtype("bool")
assert availability_readback["decision"].dtype == pd.StringDtype()
pd.testing.assert_frame_equal(availability_readback, availability)

print(availability)
print("Wrote:", AVAILABILITY_OUTPUT_PATH)
print("Availability SHA-256:", sha256(availability_bytes).hexdigest())


In [ ]:
chronological_blocks = prepared.copy()
chronological_blocks["block"] = pd.Series(
    np.where(
        chronological_blocks["observed_at"].lt(prediction_timestamp),
        "earlier",
        "later_holdout",
    ),
    dtype="string",
)

earlier_block = chronological_blocks.loc[
    chronological_blocks["block"].eq("earlier")
]
later_holdout = chronological_blocks.loc[
    chronological_blocks["block"].eq("later_holdout")
]
assert chronological_blocks.shape == (10, 5)
assert chronological_blocks["station"].dtype == pd.StringDtype()
assert str(chronological_blocks["observed_at"].dtype) == "datetime64[us, UTC]"
assert chronological_blocks["temperature_c"].dtype == np.dtype("float64")
assert chronological_blocks["source_row"].dtype == np.dtype("int64")
assert chronological_blocks["block"].dtype == pd.StringDtype()
assert len(earlier_block) == 7
assert len(later_holdout) == 3
assert set(earlier_block["station"]) == {"north", "south"}
assert set(later_holdout["station"]) == {"north", "south"}
assert earlier_block["observed_at"].max() == pd.Timestamp(
    "2026-01-15 20:00", tz="UTC"
)
assert later_holdout["observed_at"].min() == prediction_timestamp
assert earlier_block["observed_at"].max() < later_holdout[
    "observed_at"
].min()

BLOCKS_OUTPUT_PATH = OUTPUT_DIRECTORY / "chronological_blocks.csv"
blocks_bytes = write_verified_csv(
    chronological_blocks,
    BLOCKS_OUTPUT_PATH,
    expected_size=535,
    expected_sha256="7ea9752756ef882dbe19318bfbb1614c33ff6bbca45a5bbc5effe4bcad065a67",
)
blocks_readback = pd.read_csv(
    BLOCKS_OUTPUT_PATH,
    dtype={
        "station": "string",
        "observed_at": "string",
        "temperature_c": "float64",
        "source_row": "int64",
        "block": "string",
    },
)
restore_utc_timestamp(blocks_readback)
assert blocks_readback["station"].dtype == pd.StringDtype()
assert str(blocks_readback["observed_at"].dtype) == "datetime64[us, UTC]"
assert blocks_readback["temperature_c"].dtype == np.dtype("float64")
assert blocks_readback["source_row"].dtype == np.dtype("int64")
assert blocks_readback["block"].dtype == pd.StringDtype()
pd.testing.assert_frame_equal(blocks_readback, chronological_blocks)

demo3_verified = True
print(chronological_blocks)
print("Wrote:", BLOCKS_OUTPUT_PATH)
print("Blocks SHA-256:", sha256(blocks_bytes).hexdigest())


## Interpret the handoff

The two trailing means answer different questions on irregular data; neither is universally "the rolling mean." The availability inventory rejects two candidates because their latest required timestamp is later than the supplied prediction time. The later block is chronologically plausible, but it is not yet a formal training or evaluation design.


In [ ]:
assert demo3_verified is True
assert sha256(FIXTURE_PATH.read_bytes()).hexdigest() == EXPECTED_FIXTURE_SHA256
assert sha256(FEATURES_OUTPUT_PATH.read_bytes()).hexdigest() == "5a6524e8dbb37da3cc056cc648e5ff444c12cb485214bef1a95c3a67a22af3ab"
assert sha256(AVAILABILITY_OUTPUT_PATH.read_bytes()).hexdigest() == "d4125def8dcf8e23b9f33574f1dd9e14a5ed3f92889f88b455509110ad87e505"
assert sha256(BLOCKS_OUTPUT_PATH.read_bytes()).hexdigest() == "7ea9752756ef882dbe19318bfbb1614c33ff6bbca45a5bbc5effe4bcad065a67"
assert features.groupby(
    "station", observed=True, sort=True, dropna=True
).head(1)["temperature_lag_1"].isna().all()
assert availability["decision"].tolist() == ["keep", "keep", "reject", "reject"]
assert [len(earlier_block), len(later_holdout)] == [7, 3]
print("Lecture 09 Demo 3 fresh-execution verification passed.")
